# SAHI: YOLO26 training and sliced inference

This notebook trains a YOLO26n detector on this workspace's train/validation splits, evaluates it on the held-out test split, and demonstrates SAHI sliced inference on a test image. SAHI is used here for inference-time slicing; model training itself is performed by Ultralytics YOLO. The test split is never used for training.

In [1]:
%pip install ultralytics sahi

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 3.1 MB/s eta 0:00:01
   ----------------------------- ---------- 1.3/1.8 MB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import torch
import yaml

ROOT = Path.cwd()
if not (ROOT / "data.yaml").is_file() and (ROOT.parent / "data.yaml").is_file():
    ROOT = ROOT.parent

DATA_YAML = ROOT / "data.yaml"
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"Could not find data.yaml from {Path.cwd()}")

with DATA_YAML.open(encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

for split in ("train", "val", "test"):
    image_dir = Path(data_config[split])
    if not image_dir.is_absolute():
        image_dir = DATA_YAML.parent / image_dir
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Missing {split} images: {image_dir}")
    print(f"{split}: {len(list(image_dir.glob('*')))} images")

DEVICE = 0 if torch.cuda.is_available() else "cpu"
SAHI_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}; CUDA available: {torch.cuda.is_available()}; device: {DEVICE}")

train: 2099 images
val: 600 images
test: 300 images
PyTorch 2.15.0+cpu; CUDA available: False; device: cpu


## Train YOLO26n

This starts from pretrained `yolo26n.pt` and fine-tunes for 50 epochs at 640px. If training is interrupted, run the cell again to resume from the latest saved `last.pt` checkpoint (the last completed epoch). CPU training can take several hours. Results and the best checkpoint are written to `runs/detect/yolo26n_sahi`.

In [4]:
from pathlib import Path
import torch
from ultralytics import YOLO

ROOT = Path.cwd()
if not (ROOT / "data.yaml").is_file() and (ROOT.parent / "data.yaml").is_file():
    ROOT = ROOT.parent
DATA_YAML = ROOT / "data.yaml"
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"Could not find data.yaml from {Path.cwd()}")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
RUN_DIR = ROOT / "runs" / "detect" / "yolo26n_sahi"
LAST_CHECKPOINT = RUN_DIR / "weights" / "last.pt"

if LAST_CHECKPOINT.is_file():
    print(f"Resuming from {LAST_CHECKPOINT}")
    model = YOLO(str(LAST_CHECKPOINT))
    train_results = model.train(resume=True)
else:
    print("No resume checkpoint found; starting from pretrained yolo26n.pt")
    model = YOLO("yolo26n.pt")
    train_results = model.train(
        data=str(DATA_YAML),
        epochs=50,
        imgsz=640,
        batch=8,
        device=DEVICE,
        workers=0,
        project=str(ROOT / "runs" / "detect"),
        name="yolo26n_sahi",
        exist_ok=True,
        patience=10,
        seed=42,
        plots=True,
    )
print(f"Training outputs: {train_results.save_dir}")

Resuming from d:\AI-TOD.v2i.yolo26\runs\detect\yolo26n_sahi\weights\last.pt
Ultralytics 8.4.211  Python-3.13.3 torch-2.15.0+cpu CPU (13th Gen Intel Core i5-1335U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\AI-TOD.v2i.yolo26\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.001111, lrf=0.01, mask_ratio=4, max_det=1499, mixup=0.0, mode=train, model=d:\AI-TOD.v2i.yolo26\runs\detect\yo

KeyboardInterrupt: 

## Evaluate the held-out test split

First evaluate the best checkpoint with standard full-image YOLO inference. This provides a baseline for comparison with SAHI sliced inference.

In [ ]:
best_weights = Path(train_results.save_dir) / "weights" / "best.pt"
if not best_weights.is_file():
    raise FileNotFoundError(f"Best checkpoint was not created: {best_weights}")

best_model = YOLO(str(best_weights))
standard_test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=8,
    device=DEVICE,
    workers=0,
)
print(standard_test_metrics.results_dict)

## Run SAHI sliced inference

The example below splits one held-out test image into overlapping 320×320 tiles, predicts on those tiles, and merges overlapping detections. This can help with small objects, at the cost of slower inference. The output visualization is saved under `runs/sahi`.

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

test_dir = Path(data_config["test"])
if not test_dir.is_absolute():
    test_dir = DATA_YAML.parent / test_dir
test_images = sorted(p for p in test_dir.iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"})
if not test_images:
    raise FileNotFoundError(f"No test images found in {test_dir}")

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=str(best_weights),
    confidence_threshold=0.25,
    device=SAHI_DEVICE,
)

sample_image = test_images[0]
sahi_result = get_sliced_prediction(
    str(sample_image),
    detection_model,
    slice_height=320,
    slice_width=320,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    perform_standard_pred=True,
)

sahi_output_dir = ROOT / "runs" / "sahi"
sahi_output_dir.mkdir(parents=True, exist_ok=True)
sahi_result.export_visuals(export_dir=str(sahi_output_dir))
print(f"Image: {sample_image.name}")
print(f"SAHI detections: {len(sahi_result.object_prediction_list)}")
print(f"Visualization saved in: {sahi_output_dir}")
for detection in sahi_result.object_prediction_list:
    print(detection.category.name, round(detection.score.value, 3), detection.bbox.to_xyxy())